# Lesson 10 - 本番環境におけるAIエージェント

このレッスンでは、Microsoft Agent Framework（Python）を使用したAIエージェントの<strong>本番環境パターン</strong>について学びます。内容は以下の通りです：

- <strong>可観測性</strong> — エージェントのインタラクションにタイミングとログを追加する
- <strong>評価</strong> — 評価エージェントを用いて応答の品質をスコアリングする
- <strong>コスト管理</strong> — トークン最適化とモデル選択の戦略

シナリオは、旅行計画を支援する<strong>旅行代理店</strong>で、監視と評価が重ねられています。


## セットアップ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 本番環境での考慮事項

AIエージェントを試作から本番運用へ移行するには、次の3つの柱に細心の注意を払う必要があります。

1. <strong>可観測性</strong> — エージェントが何をしているか、どれくらい時間がかかっているか、どのツールを呼び出しているかを可視化する必要があります。トレーシングやロギングがなければ、本番環境の問題をデバッグすることはほぼ不可能です。

2. <strong>評価</strong> — 自動化された品質チェックにより、エージェントの応答が時間経過とともに正確で完全かつ有用なままであることを保証します。評価エージェントが定義された基準に対して応答をスコアリングできます。

3. <strong>コスト管理</strong> — トークンの使用量はコストに直接影響します。プロンプトの最適化、モデルの選択、キャッシュなどの戦略により、品質を犠牲にすることなく費用を抑制できます。


## オブザーバブルエージェントの構築

旅行ツールを定義し、レイテンシを監視できるようにエージェント呼び出しにタイミングを付加します。本番環境では、OpenTelemetryや同様のトレーシングバックエンドと統合します。


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## 評価パターン

一般的な実運用パターンでは、二番目のエージェントを<strong>評価者</strong>として使用します。評価者は主エージェントの応答を、完全性、正確性、役立ち度などの事前定義された基準に照らして評価します。

これにより以下が可能になります：
- 応答がユーザーに到達する前の自動品質ゲート
- プロンプトやモデルの変更時のリグレッション検出
- エージェントのパフォーマンスを長期間にわたり継続的に監視


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## コスト管理戦略

コスト管理は、プロダクションAIエージェントにとって重要です。主な戦略は以下の通りです:

| 戦略 | 説明 |
|---|---|
| <strong>プロンプト最適化</strong> | システム指示は簡潔に保ちます。重複するコンテキストを削除して入力トークンを減らします。 |
| <strong>モデル選択</strong> | 分類や抽出などの簡単なタスクには小型で安価なモデル（例: GPT-4o-mini）を使用し、複雑な推論には大型モデルを使います。 |
| <strong>キャッシュ</strong> | ツールの結果や頻繁なクエリをキャッシュして、冗長なAPI呼び出しを避けます。 |
| <strong>トークン予算</strong> | `max_tokens` 制限を設定して、想定外の長い応答を防ぎます。 |
| <strong>バッチ処理</strong> | 可能な限り複数のユーザークエリを1つのAPI呼び出しにまとめます。 |

実際には階層的なアプローチが効果的です。単純なリクエストは高速で安価なモデルに振り分け、複雑なクエリのみ高度なモデルにエスカレーションします。


## Summary

このレッスンでは以下のことを学びました：

1. タイミングやログを使ってエージェントのインタラクションに<strong>可観測性を追加</strong>し、トレーシングやモニタリングの基盤を築く方法。
2. 完全性、正確性、および有用性を評価する評価エージェントを使用してエージェントの応答を<strong>自動的に評価</strong>する方法。
3. プロンプトの最適化、モデル選択、キャッシュ、トークン予算を通じて<strong>コストを管理</strong>する方法。

これらの本番運用パターンは、AIエージェントをスケールした際に信頼性が高く、測定可能で、コスト効率が良いものにするのに役立ちます。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
